In [ ]:
import os
import sys
from pathlib import Path
import importlib
import torch

In [ ]:
#@title Setup
root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}
mount_drive = True  #@param {type:"boolean"}
clone_repo = False  #@param {type:"boolean"}
download_results = False  #@param {type:"boolean"}
run_training = False  #@param {type:"boolean"}

import sys
from pathlib import Path

REPO_URL = "https://github.com/TAX2310/Flood-Mapping.git"

if not mount_drive and not clone_repo:
    raise ValueError("Either mount_drive or clone_repo must be True.")

if mount_drive:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    root_path = "Flood-Mapping"

repo = Path(root_path)

if clone_repo and not repo.exists():
    !git clone $REPO_URL $root_path

assert repo.exists(), f"Repo not found at {repo}. Enable clone_repo or fix root_path."

sys.path.append(str(repo))
from src.config import S1_CFG

cfg = S1_CFG()
cfg.ROOT = repo
cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
requirements = cfg.ROOT / "requirements.txt"
!pip install -r {requirements}

In [ ]:
import src.data.sturm_fusion as SturmFusion

data_root = SturmFusion.download_and_extract_dataset(cfg)

img_dir = cfg.S1_PATH
mask_dir = cfg.MASK_PATH

print("Image dir exists:", img_dir.exists())
print("Mask dir exists:", mask_dir.exists())

print("Num images:", len(list(img_dir.glob("*.tif"))))
print("Num masks:", len(list(mask_dir.glob("*.tif"))))

if download_results:
    SturmFusion.download_and_extract_results(cfg)

In [ ]:
import src.train.training as training
import src.test.testing as testing
import src.util.io as io
import src.util.plotting as plot

In [ ]:
if run_training:
    for learning_rate in cfg.LEARNING_RATES:
        for batch_size in cfg.BATCH_SIZES:
            for weight_decay in cfg.WEIGHT_DECAYS:
                for dropout_rate in cfg.DROPOUT_RATES:
                    cfg.LR = learning_rate
                    cfg.BATCH_SIZE = batch_size
                    cfg.WEIGHT_DECAY = weight_decay
                    cfg.DROPOUT_RATE = dropout_rate
                    training.train_from_file(cfg, num_workers=num_workers)

In [ ]:
plot.plot_hp_comparison_bar(cfg, save_path=cfg.FIG_EXPORTS_DIR/"s1_hp_iou_f1.pdf")

In [ ]:
plot.view_training_metrics(cfg)

In [ ]:
testing.test_model(cfg, cfg.S1_MODEL)

In [ ]:
testing.select_model_to_test(cfg)

In [ ]:
import src.inference.inference as inference

samples = ["EMSR570_AOI02_07_03_2_1.tif", "EMSR470_AOI01_46_07_2_1.tif", "EMSR470_AOI01_47_10_1_2.tif"]

#samples = ["EMSR470_AOI01_46_07_2_1.tif","EMSR441_AOI05_2_3_2_2.tif","EMSR570_AOI02_07_03_2_1.tif"]

#samples = ["EMSR470_AOI01_29_13_2_2.tif", "EMSR407_AOI01_03_17_2_1.tif", "EMSR470_AOI01_10_25_1_1.tif", "EMSR470_AOI01_47_10_1_2.tif", "EMSR629_AOI01_09_01_1_2.tif"]

results = inference.inference(cfg, cfg.S1_MODEL, samples)
#all_results = inference.inference(cfg, cfg.S1_MODEL)
#io.create_inference_results_csv(cfg, all_results, cfg.METADATA_CSV, cfg.S1_TEST_RESULTS_CSV)

In [ ]:
plot.plot_sample_results(cfg, results)

In [ ]:
plot.plot_metric_distribution_from_csv(cfg.S1_TEST_RESULTS_CSV, save_path=cfg.FIG_EXPORTS_DIR/"s1_iou_dist.pdf")

In [ ]:
plot.plot_iou_vs_flood_scatter([cfg.S1_TEST_RESULTS_CSV], save_path=cfg.FIG_EXPORTS_DIR/"s1_scatter.pdf", show_legend=False)

In [ ]:
plot.plot_average_iou_per_event(cfg.S1_TEST_RESULTS_CSV, save_path=cfg.FIG_EXPORTS_DIR/"s1_iou_per_event.pdf")